# Camera-POV Retraining for AFM Relocation

This notebook stages camera-FOV site memories and retrains the relocation models from that staged dataset.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
from pathlib import Path

def looks_like_repo(path: Path) -> bool:
    return (path / 'requirements.txt').exists() and (path / 'training').exists()

def find_repo_candidates(root: Path, limit=20):
    matches = []
    if not root.exists():
        return matches
    for req in root.rglob('requirements.txt'):
        candidate = req.parent
        if looks_like_repo(candidate):
            matches.append(candidate)
            if len(matches) >= limit:
                break
    return matches

candidate_roots = [
    Path('/content/drive/MyDrive/AFM-Hysteresis-Simulation'),
    Path('/content/drive/MyDrive/Colab Notebooks/AFM-Hysteresis-Simulation'),
    Path('/content/AFM-Hysteresis-Simulation'),
    Path('/content/drive/MyDrive'),
    Path.cwd(),
]

repo = None
for root in candidate_roots:
    if root.exists() and looks_like_repo(root):
        repo = root
        break
    matches = find_repo_candidates(root)
    if matches:
        repo = matches[0]
        break

if repo is None:
    raise FileNotFoundError('Could not locate repo root in Colab Drive runtime.')

os.chdir(repo)
REPO_DIR = str(repo)
print('REPO_DIR =', REPO_DIR)


REPO_DIR = /content/drive/MyDrive/AFM-Hysteresis-Simulation


In [3]:
%cd {REPO_DIR}
!python --version
!pip install -r requirements.txt


/content
Python 3.12.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.3 MB/s eta 0:00:00


In [4]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Selected device =', DEVICE)
print('torch.cuda.is_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu_name =', torch.cuda.get_device_name(0))


Selected device = cuda
torch.cuda.is_available = True
gpu_name = Tesla T4


In [5]:
!python training/run_camera_pov_retraining.py --device {DEVICE}


Manifest written to: /content/drive/MyDrive/AFM-Hysteresis-Simulation/collected_data/prepared_training/camera_fov_training_manifest.csv
Total site memories: 49
Camera-FOV-ready site memories: 19
Legacy site memories: 30
Staged camera-only site memories: 19
Staged root: /content/drive/MyDrive/AFM-Hysteresis-Simulation/collected_data/prepared_training/site_memories_camera_only

Training Phase 2 camera-only models...
[ WARN:0@233.535] global loadsave.cpp:278 findDecoder imread_('/content/drive/MyDrive/AFM-Hysteresis-Simulation/collected_data/prepared_training/site_memories_camera_only/afm_ideal_scan/20260816_132705_unlabeled_site/landmarks\lowmag\lowmag_01.png'): can't open/read file: check file path/integrity
[ WARN:0@233.536] global loadsave.cpp:278 findDecoder imread_('/content/drive/MyDrive/AFM-Hysteresis-Simulation/collected_data/prepared_training/site_memories_camera_only/afm_ideal_scan/20260816_132705_unlabeled_site/landmarks\lowmag\lowmag_02.png'): can't open/read file: check file

In [6]:
!ls -lh collected_data/models
!find collected_data/prepared_training -maxdepth 2 -type f | sort


total 7.7M
-rw------- 1 root root 5.1M Aug 16 14:01 deep_remount_predictor_real.pkl
-rw------- 1 root root 310K Aug 16 14:01 lowmag_embedding_index.pkl
-rw------- 1 root root 2.2M Aug 16 14:01 remount_transform_predictor.pkl
-rw------- 1 root root 239K Aug 16 14:00 same_site_classifier.pkl
collected_data/prepared_training/camera_fov_training_manifest.csv
